In [54]:
import os
import tqdm
import numpy as np
import pandas as pd
from tushare import pro_api, set_token
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time

In [2]:
set_token('b549c6e18f71105519447d872727cc2b7f0022f071c9eb27d0256ebb')
pro = pro_api()

### 交易日历

In [ ]:
#trade_cal = pro.trade_cal()
#trade_cal.to_csv('data/trade_cal.csv', index=False)
trade_cal = pd.read_csv('data/trade_cal.csv', dtype={'cal_date': str})
data_cal = trade_cal[(trade_cal['cal_date']>='20051230')&(trade_cal['cal_date']<'20260101')]

### 日线

In [7]:
data_path = 'data/daily_K/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.daily(trade_date=d['cal_date'])
        adj_factor = pro.adj_factor(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['pre_close', 'change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'stock-{d["cal_date"]}.ftr'))
    else:
        continue

2924it [11:48,  4.13it/s]


### 每日基础数据

In [8]:
data_path = 'data/daily_basic/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily_basic = pro.daily_basic(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        daily_basic.to_feather(os.path.join(data_path, f'basic-{d["cal_date"]}.ftr'))
    else:
        continue

2924it [11:37,  4.19it/s]


In [ ]:
#industry = pro.index_member_all()
#industry.to_csv('data/industry.csv', index=False)
#allstock = pro.stock_basic()
#allstock.to_csv('data/allstock.csv', index=False)

### 股票列表

In [10]:
allstock = pd.read_csv('data/allstock.csv')
allstock_list = allstock.ts_code.tolist()
fins = []

In [14]:
len(fins)

3200

### 财务指标

In [ ]:
batch_size = 200
with tqdm.tqdm(total=len(allstock_list), desc="处理股票", unit="stock") as pbar:
    for i in range(0, len(allstock_list), batch_size):
        batch = allstock_list[i:i + batch_size]
        for d in batch:
            tmp = pro.fina_indicator(ts_code=d, start_date='20050930', end_date='20130929').drop_duplicates(subset=['ts_code', 'end_date'], keep='first')
            fins.append(tmp)
            pbar.update(1)
        
        # 每批次结束后休息
        if i + batch_size < len(allstock_list):
            time.sleep(35)

处理股票: 100%|██████████| 2286/2286 [11:46<00:00,  3.23stock/s]  


In [16]:
df_fins = pd.concat(fins)
for d in df_fins.end_date.unique():
    df_fins[df_fins['end_date'] == d].to_feather(os.path.join('data/finance/', f'fina-{d}.ftr'))

### 资金流向&龙虎榜&融资融券

In [ ]:
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        moneyflow = pro.moneyflow(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        moneyflow['trade_date'] = pd.to_datetime(moneyflow['trade_date'])
        moneyflow.to_feather(os.path.join('data/moneyflow/', f'moneyflow-{d["cal_date"]}.ftr'))
        toplist = pro.top_list(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        toplist['trade_date'] = pd.to_datetime(toplist['trade_date'])
        toplist.to_feather(os.path.join('data/toplist/', f'toplist-{d["cal_date"]}.ftr'))
        tmp = pro.margin_detail(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        tmp['trade_date'] = pd.to_datetime(tmp['trade_date'])
        tmp.to_feather(os.path.join('data/rzrq/', f'margin-{d["cal_date"]}.ftr'))
    else:
        continue

3237it [19:55,  2.79it/s]

### 宏观

In [ ]:
cpi = pro.cn_cpi(start_m='200512', end_m='202512')
ppi = pro.cn_ppi(start_m='200512', end_m='202512')
money = pro.cn_m(start_m='200512', end_m='202512')
sf = pro.sf_month(start_m='200512', end_m='202512')

df_macro = cpi.merge(ppi, on='month', how='left').merge(money, on='month', how='left').merge(sf, on='month', how='left')
df_macro['month'] = pd.to_datetime(df_macro['month'], format='%Y%m')
df_macro.to_csv(os.path.join('data/', 'macro.csv'), index=False)

### 指数相关

In [36]:
index_list = [
    '000300.SH',   # 沪深300
    '000001.SH',     # 上证指数
    '399001.SZ',     # 深证成指
    '000905.SH',   # 中证500
    '399006.SZ',   # 创业板指
    '000016.SH',   # 上证50
    '000688.SH',   # 科创50
    '000852.SH',   # 中证1000
]

for d in tqdm.tqdm(index_list):
    index = pro.index_daily(ts_code=d, start_date='20051230', end_date='20251231')
    index['trade_date'] = pd.to_datetime(index['trade_date'])
    index.to_csv(os.path.join('data/index/index_daily_K/', f'{d}.csv'), index=False)
    indexbasic = pro.index_dailybasic(ts_code=d, start_date='20051230', end_date='20251231')
    indexbasic['trade_date'] = pd.to_datetime(indexbasic['trade_date'])
    indexbasic.to_csv(os.path.join('data/index/index_daily_basic/', f'{d}_basic.csv'), index=False)


100%|██████████| 8/8 [00:04<00:00,  1.65it/s]


### 股东增减持

In [63]:
current = datetime.strptime('20220801', '%Y%m%d')
with tqdm.tqdm() as pbar:
    while current <= datetime.strptime('20251231', '%Y%m%d'):
        # 当月第一天
        month_start = current.replace(day=1)
        # 当月最后一天
        month_end = (month_start + relativedelta(months=1) - relativedelta(days=1))
        df_hodlers =  pro.stk_holdertrade(start_date = month_start.strftime('%Y%m%d'), end_date = month_end.strftime('%Y%m%d'))
        if len(df_hodlers)>=3000:
            print(f"Warning: {month_start.strftime('%Y-%m')} may exceed API limits.")
        else:
            df_hodlers.to_feather(os.path.join('data/holdertrade/', f'holders-{month_start.strftime("%Y%m")}.ftr'))
        pbar.update(1)
        # 进入下一月
        current = month_start + relativedelta(months=1)


41it [00:06,  5.94it/s]
